# Complete Classification Algorithms Comparison

**Standalone Google Colab Notebook - Everything Included!**

This notebook contains everything you need - just upload to Google Colab and run all cells!

**Algorithms Implemented from Scratch:**
- k-NN Classifier (Euclidean/Manhattan, weighted voting)
- Random Forest (Decision Trees with Gini/Entropy)  
- Naive Bayes (Gaussian for continuous, frequency-based for categorical)

**Dataset:** Heart Disease UCI (will auto-download)

---

## Step 1: Install Dependencies

In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn scipy

## Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## Step 3: Create Python Module Files

The following cells create all necessary Python files with our custom implementations.

### Create knn_classifier.py

In [ ]:
%%writefile knn_classifier.py
"""
k-NN Classifier Implementation from Scratch

Implements k-Nearest Neighbors classifier with:
- Euclidean and Manhattan distance metrics
- Weighted voting based on distance
- Configurable k values
"""

import numpy as np
from collections import Counter
from scipy.spatial.distance import euclidean, cityblock


class KNNClassifier:
    """
    k-Nearest Neighbors Classifier implemented from scratch.
    """
    
    def __init__(self, k=5, distance_metric='euclidean', weighted=True):
        """
        Initialize k-NN Classifier.
        
        Parameters:
        -----------
        k : int, default=5
            Number of neighbors to consider
        distance_metric : str, default='euclidean'
            Distance metric to use ('euclidean' or 'manhattan')
        weighted : bool, default=True
            If True, use distance-weighted voting. If False, use majority voting.
        """
        self.k = k
        self.distance_metric = distance_metric.lower()
        self.weighted = weighted
        self.X_train = None
        self.y_train = None
        
        if self.distance_metric not in ['euclidean', 'manhattan']:
            raise ValueError("distance_metric must be 'euclidean' or 'manhattan'")
    
    def _calculate_distance(self, x1, x2):
        """
        Calculate distance between two points.
        
        Parameters:
        -----------
        x1 : array-like
            First point
        x2 : array-like
            Second point
            
        Returns:
        --------
        float
            Distance between x1 and x2
        """
        if self.distance_metric == 'euclidean':
            return np.sqrt(np.sum((x1 - x2) ** 2))
        else:  # manhattan
            return np.sum(np.abs(x1 - x2))
    
    def fit(self, X, y):
        """
        Fit the k-NN classifier.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data
        y : array-like of shape (n_samples,)
            Target values
        """
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self
    
    def predict(self, X):
        """
        Predict class labels for samples.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples to predict
            
        Returns:
        --------
        array-like of shape (n_samples,)
            Predicted class labels
        """
        X = np.array(X)
        predictions = []
        
        for sample in X:
            # Calculate distances to all training samples
            distances = [self._calculate_distance(sample, x) for x in self.X_train]
            
            # Get k nearest neighbors
            k_indices = np.argsort(distances)[:self.k]
            k_distances = [distances[i] for i in k_indices]
            k_labels = [self.y_train[i] for i in k_indices]
            
            if self.weighted:
                # Weighted voting: weights are inverse of distance
                # Add small epsilon to avoid division by zero
                epsilon = 1e-10
                weights = [1 / (d + epsilon) for d in k_distances]
                
                # Calculate weighted vote for each class
                class_votes = {}
                for label, weight in zip(k_labels, weights):
                    if label not in class_votes:
                        class_votes[label] = 0
                    class_votes[label] += weight
                
                # Predict class with highest weighted vote
                prediction = max(class_votes, key=class_votes.get)
            else:
                # Majority voting
                prediction = Counter(k_labels).most_common(1)[0][0]
            
            predictions.append(prediction)
        
        return np.array(predictions)
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples to predict
            
        Returns:
        --------
        array-like of shape (n_samples, n_classes)
            Class probabilities
        """
        X = np.array(X)
        unique_classes = np.unique(self.y_train)
        probabilities = []
        
        for sample in X:
            # Calculate distances to all training samples
            distances = [self._calculate_distance(sample, x) for x in self.X_train]
            
            # Get k nearest neighbors
            k_indices = np.argsort(distances)[:self.k]
            k_distances = [distances[i] for i in k_indices]
            k_labels = [self.y_train[i] for i in k_indices]
            
            if self.weighted:
                # Weighted probabilities
                epsilon = 1e-10
                weights = [1 / (d + epsilon) for d in k_distances]
                
                class_probs = {cls: 0.0 for cls in unique_classes}
                total_weight = sum(weights)
                
                for label, weight in zip(k_labels, weights):
                    class_probs[label] += weight
                
                # Normalize
                probs = [class_probs[cls] / total_weight for cls in unique_classes]
            else:
                # Majority voting probabilities
                vote_counts = Counter(k_labels)
                total_votes = len(k_labels)
                probs = [vote_counts.get(cls, 0) / total_votes for cls in unique_classes]
            
            probabilities.append(probs)
        
        return np.array(probabilities)


### Create decision_tree.py

In [ ]:
%%writefile decision_tree.py
"""
Decision Tree Implementation from Scratch

Implements decision tree classifier with:
- Gini impurity and Entropy as split criteria
- Recursive binary splitting
- Configurable max depth
"""

import numpy as np
from collections import Counter


class DecisionTree:
    """
    Decision Tree Classifier implemented from scratch.
    """
    
    def __init__(self, max_depth=None, min_samples_split=2, criterion='gini'):
        """
        Initialize Decision Tree.
        
        Parameters:
        -----------
        max_depth : int or None, default=None
            Maximum depth of the tree
        min_samples_split : int, default=2
            Minimum number of samples required to split a node
        criterion : str, default='gini'
            Splitting criterion ('gini' or 'entropy')
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion.lower()
        self.tree = None
        
        if self.criterion not in ['gini', 'entropy']:
            raise ValueError("criterion must be 'gini' or 'entropy'")
    
    def _gini_impurity(self, y):
        """
        Calculate Gini impurity.
        
        Parameters:
        -----------
        y : array-like
            Target values
            
        Returns:
        --------
        float
            Gini impurity
        """
        if len(y) == 0:
            return 0
        
        counts = Counter(y)
        proportions = [count / len(y) for count in counts.values()]
        return 1 - sum(p ** 2 for p in proportions)
    
    def _entropy(self, y):
        """
        Calculate entropy.
        
        Parameters:
        -----------
        y : array-like
            Target values
            
        Returns:
        --------
        float
            Entropy
        """
        if len(y) == 0:
            return 0
        
        counts = Counter(y)
        proportions = [count / len(y) for count in counts.values()]
        return -sum(p * np.log2(p) if p > 0 else 0 for p in proportions)
    
    def _impurity(self, y):
        """
        Calculate impurity based on criterion.
        
        Parameters:
        -----------
        y : array-like
            Target values
            
        Returns:
        --------
        float
            Impurity value
        """
        if self.criterion == 'gini':
            return self._gini_impurity(y)
        else:  # entropy
            return self._entropy(y)
    
    def _information_gain(self, y_parent, y_left, y_right):
        """
        Calculate information gain from a split.
        
        Parameters:
        -----------
        y_parent : array-like
            Target values of parent node
        y_left : array-like
            Target values of left child
        y_right : array-like
            Target values of right child
            
        Returns:
        --------
        float
            Information gain
        """
        parent_impurity = self._impurity(y_parent)
        n = len(y_parent)
        n_left = len(y_left)
        n_right = len(y_right)
        
        if n == 0:
            return 0
        
        weighted_impurity = (n_left / n) * self._impurity(y_left) + \
                           (n_right / n) * self._impurity(y_right)
        
        return parent_impurity - weighted_impurity
    
    def _find_best_split(self, X, y):
        """
        Find the best split for a node.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Feature matrix
        y : array-like of shape (n_samples,)
            Target values
            
        Returns:
        --------
        dict
            Best split information
        """
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        for feature_idx in range(n_features):
            # Get unique values for this feature
            feature_values = np.unique(X[:, feature_idx])
            
            # Try thresholds between consecutive values
            for i in range(len(feature_values) - 1):
                threshold = (feature_values[i] + feature_values[i + 1]) / 2
                
                # Split data
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                
                y_left = y[left_mask]
                y_right = y[right_mask]
                
                # Calculate information gain
                gain = self._information_gain(y, y_left, y_right)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return {
            'feature': best_feature,
            'threshold': best_threshold,
            'gain': best_gain
        }
    
    def _majority_class(self, y):
        """
        Get the majority class.
        
        Parameters:
        -----------
        y : array-like
            Target values
            
        Returns:
        --------
            Majority class label
        """
        counts = Counter(y)
        return counts.most_common(1)[0][0]
    
    def _build_tree(self, X, y, depth=0):
        """
        Recursively build the decision tree.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Feature matrix
        y : array-like of shape (n_samples,)
            Target values
        depth : int, default=0
            Current depth of the tree
            
        Returns:
        --------
        dict
            Tree node
        """
        n_samples = len(y)
        n_classes = len(np.unique(y))
        
        # Stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           (n_samples < self.min_samples_split) or \
           (n_classes == 1):
            return {
                'leaf': True,
                'class': self._majority_class(y),
                'samples': n_samples
            }
        
        # Find best split
        best_split = self._find_best_split(X, y)
        
        # If no good split found, create leaf
        if best_split['gain'] == 0 or best_split['feature'] is None:
            return {
                'leaf': True,
                'class': self._majority_class(y),
                'samples': n_samples
            }
        
        # Split data
        left_mask = X[:, best_split['feature']] <= best_split['threshold']
        right_mask = ~left_mask
        
        X_left = X[left_mask]
        y_left = y[left_mask]
        X_right = X[right_mask]
        y_right = y[right_mask]
        
        # Recursively build left and right subtrees
        left_child = self._build_tree(X_left, y_left, depth + 1)
        right_child = self._build_tree(X_right, y_right, depth + 1)
        
        return {
            'leaf': False,
            'feature': best_split['feature'],
            'threshold': best_split['threshold'],
            'gain': best_split['gain'],
            'left': left_child,
            'right': right_child,
            'samples': n_samples
        }
    
    def fit(self, X, y):
        """
        Fit the decision tree.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data
        y : array-like of shape (n_samples,)
            Target values
        """
        X = np.array(X)
        y = np.array(y)
        self.tree = self._build_tree(X, y)
        return self
    
    def _predict_sample(self, sample, node):
        """
        Predict class for a single sample.
        
        Parameters:
        -----------
        sample : array-like
            Sample to predict
        node : dict
            Current node in the tree
            
        Returns:
        --------
            Predicted class label
        """
        if node['leaf']:
            return node['class']
        
        if sample[node['feature']] <= node['threshold']:
            return self._predict_sample(sample, node['left'])
        else:
            return self._predict_sample(sample, node['right'])
    
    def predict(self, X):
        """
        Predict class labels for samples.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples to predict
            
        Returns:
        --------
        array-like of shape (n_samples,)
            Predicted class labels
        """
        X = np.array(X)
        predictions = [self._predict_sample(sample, self.tree) for sample in X]
        return np.array(predictions)
    
    def get_feature_importance(self, feature_names=None):
        """
        Calculate feature importance based on information gain.
        
        Parameters:
        -----------
        feature_names : array-like, optional
            Names of features
            
        Returns:
        --------
        dict
            Feature importance scores
        """
        def calculate_importance(node, total_samples):
            importance = {}
            if not node['leaf']:
                # Calculate importance as weighted gain
                feature_idx = node['feature']
                gain = node['gain']
                samples = node['samples']
                
                if feature_names:
                    feature_name = feature_names[feature_idx]
                else:
                    feature_name = f'feature_{feature_idx}'
                
                importance[feature_name] = gain * (samples / total_samples)
                
                # Recursively calculate for children
                if 'left' in node:
                    left_importance = calculate_importance(node['left'], total_samples)
                    for key, value in left_importance.items():
                        importance[key] = importance.get(key, 0) + value
                
                if 'right' in node:
                    right_importance = calculate_importance(node['right'], total_samples)
                    for key, value in right_importance.items():
                        importance[key] = importance.get(key, 0) + value
            
            return importance
        
        if self.tree is None:
            return {}
        
        total_samples = self.tree['samples']
        return calculate_importance(self.tree, total_samples)


### Create random_forest.py

In [ ]:
%%writefile random_forest.py
"""
Random Forest Implementation from Scratch

Implements Random Forest classifier with:
- Bootstrap sampling (bagging)
- Multiple decision trees
- Majority voting for predictions
"""

import numpy as np
from decision_tree import DecisionTree
from collections import Counter


class RandomForest:
    """
    Random Forest Classifier implemented from scratch.
    """
    
    def __init__(self, n_trees=5, max_depth=None, min_samples_split=2, 
                 criterion='gini', random_state=None):
        """
        Initialize Random Forest.
        
        Parameters:
        -----------
        n_trees : int, default=5
            Number of trees in the forest
        max_depth : int or None, default=None
            Maximum depth of each tree
        min_samples_split : int, default=2
            Minimum number of samples required to split a node
        criterion : str, default='gini'
            Splitting criterion ('gini' or 'entropy')
        random_state : int or None, default=None
            Random seed for reproducibility
        """
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion
        self.random_state = random_state
        self.trees = []
        
        if random_state is not None:
            np.random.seed(random_state)
    
    def _bootstrap_sample(self, X, y):
        """
        Create a bootstrap sample of the data.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Feature matrix
        y : array-like of shape (n_samples,)
            Target values
            
        Returns:
        --------
        tuple
            Bootstrap sampled X and y
        """
        n_samples = len(y)
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        return X[indices], y[indices]
    
    def fit(self, X, y):
        """
        Fit the Random Forest.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data
        y : array-like of shape (n_samples,)
            Target values
        """
        X = np.array(X)
        y = np.array(y)
        
        self.trees = []
        
        for i in range(self.n_trees):
            # Create bootstrap sample
            X_boot, y_boot = self._bootstrap_sample(X, y)
            
            # Train decision tree
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
        
        return self
    
    def predict(self, X):
        """
        Predict class labels for samples using majority voting.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples to predict
            
        Returns:
        --------
        array-like of shape (n_samples,)
            Predicted class labels
        """
        X = np.array(X)
        predictions = []
        
        for sample in X:
            # Get predictions from all trees
            tree_predictions = [tree.predict([sample])[0] for tree in self.trees]
            
            # Majority voting
            prediction = Counter(tree_predictions).most_common(1)[0][0]
            predictions.append(prediction)
        
        return np.array(predictions)
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples to predict
            
        Returns:
        --------
        array-like of shape (n_samples, n_classes)
            Class probabilities
        """
        X = np.array(X)
        all_classes = set()
        
        # Collect all unique classes from training
        for tree in self.trees:
            if hasattr(tree, 'y_train'):
                all_classes.update(tree.y_train)
        
        if not all_classes:
            # If we don't have access to classes, use predictions to infer
            sample_preds = self.predict(X)
            all_classes = set(sample_preds)
        
        all_classes = sorted(list(all_classes))
        n_classes = len(all_classes)
        n_samples = len(X)
        
        probabilities = np.zeros((n_samples, n_classes))
        
        for i, sample in enumerate(X):
            # Get predictions from all trees
            tree_predictions = [tree.predict([sample])[0] for tree in self.trees]
            
            # Calculate probability as fraction of trees voting for each class
            for j, cls in enumerate(all_classes):
                probabilities[i, j] = tree_predictions.count(cls) / len(self.trees)
        
        return probabilities
    
    def get_feature_importance(self, feature_names=None):
        """
        Calculate feature importance by averaging across all trees.
        
        Parameters:
        -----------
        feature_names : array-like, optional
            Names of features
            
        Returns:
        --------
        dict
            Average feature importance scores
        """
        all_importances = []
        
        for tree in self.trees:
            tree_importance = tree.get_feature_importance(feature_names)
            all_importances.append(tree_importance)
        
        # Average importances across all trees
        avg_importance = {}
        for importance_dict in all_importances:
            for feature, value in importance_dict.items():
                avg_importance[feature] = avg_importance.get(feature, 0) + value
        
        # Normalize
        n_trees = len(self.trees)
        for feature in avg_importance:
            avg_importance[feature] /= n_trees
        
        return avg_importance


### Create naive_bayes.py

In [ ]:
%%writefile naive_bayes.py
"""
Naive Bayes Implementation from Scratch

Implements Naive Bayes classifier with:
- Gaussian Naive Bayes for continuous features
- Frequency-based probability estimation for categorical features
"""

import numpy as np
from collections import Counter, defaultdict


class NaiveBayes:
    """
    Naive Bayes Classifier implemented from scratch.
    Supports both continuous (Gaussian) and categorical features.
    """
    
    def __init__(self, feature_types=None):
        """
        Initialize Naive Bayes classifier.
        
        Parameters:
        -----------
        feature_types : list of str, optional
            List specifying type of each feature: 'continuous' or 'categorical'
            If None, all features are assumed to be continuous
        """
        self.feature_types = feature_types
        self.classes_ = None
        self.class_prior_ = None
        self.feature_params_ = None
    
    def _calculate_gaussian_params(self, X):
        """
        Calculate mean and standard deviation for Gaussian distribution.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples,)
            Feature values for one feature
            
        Returns:
        --------
        tuple
            (mean, std) for Gaussian distribution
        """
        mean = np.mean(X)
        std = np.std(X)
        # Add small epsilon to avoid zero std
        std = max(std, 1e-10)
        return mean, std
    
    def _gaussian_pdf(self, x, mean, std):
        """
        Calculate Gaussian probability density function.
        
        Parameters:
        -----------
        x : float
            Feature value
        mean : float
            Mean of Gaussian distribution
        std : float
            Standard deviation of Gaussian distribution
            
        Returns:
        --------
        float
            Probability density
        """
        coefficient = 1 / (std * np.sqrt(2 * np.pi))
        exponent = -0.5 * ((x - mean) / std) ** 2
        return coefficient * np.exp(exponent)
    
    def _calculate_categorical_params(self, X):
        """
        Calculate probability distribution for categorical feature.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples,)
            Feature values for one feature
            
        Returns:
        --------
        dict
            Probability for each category (with Laplace smoothing)
        """
        counts = Counter(X)
        total = len(X)
        
        # Laplace smoothing: add 1 to each count
        unique_values = set(X)
        probabilities = {}
        
        for value in unique_values:
            # Laplace smoothing: (count + 1) / (total + n_unique)
            probabilities[value] = (counts[value] + 1) / (total + len(unique_values))
        
        return probabilities
    
    def _categorical_prob(self, x, prob_dist):
        """
        Get probability for a categorical value.
        
        Parameters:
        -----------
        x : any
            Feature value
        prob_dist : dict
            Probability distribution for categories
            
        Returns:
        --------
        float
            Probability
        """
        if x in prob_dist:
            return prob_dist[x]
        else:
            # If value not seen in training, use Laplace smoothing
            return 1 / (sum(prob_dist.values()) * len(prob_dist) + 1)
    
    def fit(self, X, y):
        """
        Fit the Naive Bayes classifier.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data
        y : array-like of shape (n_samples,)
            Target values
        """
        X = np.array(X)
        y = np.array(y)
        
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        
        # Determine feature types if not provided
        if self.feature_types is None:
            # Auto-detect: assume continuous for numeric, categorical for non-numeric
            self.feature_types = []
            for i in range(n_features):
                if np.issubdtype(X[:, i].dtype, np.number):
                    # Check if values look discrete (few unique values relative to samples)
                    unique_vals = len(np.unique(X[:, i]))
                    if unique_vals < 10 and unique_vals < len(X) * 0.1:
                        self.feature_types.append('categorical')
                    else:
                        self.feature_types.append('continuous')
                else:
                    self.feature_types.append('categorical')
        
        # Calculate class prior probabilities
        class_counts = Counter(y)
        total_samples = len(y)
        self.class_prior_ = {cls: count / total_samples 
                            for cls, count in class_counts.items()}
        
        # Calculate feature parameters for each class
        self.feature_params_ = {}
        
        for cls in self.classes_:
            X_class = X[y == cls]
            self.feature_params_[cls] = []
            
            for feature_idx in range(n_features):
                feature_values = X_class[:, feature_idx]
                feature_type = self.feature_types[feature_idx]
                
                if feature_type == 'continuous':
                    # Gaussian parameters
                    mean, std = self._calculate_gaussian_params(feature_values)
                    self.feature_params_[cls].append({
                        'type': 'gaussian',
                        'mean': mean,
                        'std': std
                    })
                else:  # categorical
                    # Categorical probability distribution
                    prob_dist = self._calculate_categorical_params(feature_values)
                    self.feature_params_[cls].append({
                        'type': 'categorical',
                        'prob_dist': prob_dist
                    })
        
        return self
    
    def _calculate_likelihood(self, x, feature_params):
        """
        Calculate likelihood for a single feature value.
        
        Parameters:
        -----------
        x : any
            Feature value
        feature_params : dict
            Parameters for the feature distribution
            
        Returns:
        --------
        float
            Likelihood (probability)
        """
        if feature_params['type'] == 'gaussian':
            return self._gaussian_pdf(x, feature_params['mean'], feature_params['std'])
        else:  # categorical
            return self._categorical_prob(x, feature_params['prob_dist'])
    
    def _calculate_posterior(self, X, cls):
        """
        Calculate posterior probability for a class.
        
        Parameters:
        -----------
        X : array-like of shape (n_features,)
            Sample features
        cls : any
            Class label
            
        Returns:
        --------
        float
            Log posterior probability
        """
        # Start with log prior
        log_posterior = np.log(self.class_prior_[cls])
        
        # Add log likelihood for each feature (Naive Bayes assumption: independence)
        for feature_idx, feature_value in enumerate(X):
            feature_params = self.feature_params_[cls][feature_idx]
            likelihood = self._calculate_likelihood(feature_value, feature_params)
            
            # Use log to avoid numerical underflow
            log_likelihood = np.log(max(likelihood, 1e-10))
            log_posterior += log_likelihood
        
        return log_posterior
    
    def predict(self, X):
        """
        Predict class labels for samples.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples to predict
            
        Returns:
        --------
        array-like of shape (n_samples,)
            Predicted class labels
        """
        X = np.array(X)
        predictions = []
        
        for sample in X:
            # Calculate posterior for each class
            posteriors = {cls: self._calculate_posterior(sample, cls) 
                         for cls in self.classes_}
            
            # Predict class with highest posterior
            prediction = max(posteriors, key=posteriors.get)
            predictions.append(prediction)
        
        return np.array(predictions)
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples to predict
            
        Returns:
        --------
        array-like of shape (n_samples, n_classes)
            Class probabilities
        """
        X = np.array(X)
        probabilities = []
        
        for sample in X:
            # Calculate log posteriors for each class
            log_posteriors = {cls: self._calculate_posterior(sample, cls) 
                             for cls in self.classes_}
            
            # Convert to probabilities using log-sum-exp trick for numerical stability
            log_posterior_values = list(log_posteriors.values())
            max_log_posterior = max(log_posterior_values)
            
            # Normalize
            exp_log_posteriors = {cls: np.exp(log_p - max_log_posterior) 
                                 for cls, log_p in log_posteriors.items()}
            sum_exp = sum(exp_log_posteriors.values())
            
            probs = [exp_log_posteriors[cls] / sum_exp for cls in self.classes_]
            probabilities.append(probs)
        
        return np.array(probabilities)


### Create evaluation_utils.py

In [ ]:
%%writefile evaluation_utils.py
"""
Evaluation Utilities

Provides functions for calculating metrics and generating visualizations.
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, precision_recall_curve
)


def calculate_metrics(y_true, y_pred, y_proba=None, average='weighted'):
    """
    Calculate classification metrics.
    
    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    y_proba : array-like, optional
        Predicted probabilities (for ROC-AUC)
    average : str, default='weighted'
        Averaging strategy for multi-class metrics
        
    Returns:
    --------
    dict
        Dictionary of metrics
    """
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average=average, zero_division=0),
        'recall': recall_score(y_true, y_pred, average=average, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, average=average, zero_division=0)
    }
    
    # Calculate ROC-AUC if probabilities are provided and binary classification
    if y_proba is not None:
        try:
            # For binary classification
            if len(np.unique(y_true)) == 2:
                fpr, tpr, _ = roc_curve(y_true, y_proba[:, 1])
                metrics['roc_auc'] = auc(fpr, tpr)
            else:
                # Multi-class: calculate for each class and average
                n_classes = len(np.unique(y_true))
                roc_aucs = []
                for i in range(n_classes):
                    y_true_binary = (y_true == np.unique(y_true)[i]).astype(int)
                    if len(np.unique(y_true_binary)) == 2:  # Check if class exists
                        fpr, tpr, _ = roc_curve(y_true_binary, y_proba[:, i])
                        roc_aucs.append(auc(fpr, tpr))
                if roc_aucs:
                    metrics['roc_auc'] = np.mean(roc_aucs)
        except Exception as e:
            print(f"Warning: Could not calculate ROC-AUC: {e}")
            metrics['roc_auc'] = None
    
    return metrics


def plot_confusion_matrix(y_true, y_pred, class_names=None, title='Confusion Matrix', ax=None):
    """
    Plot confusion matrix.
    
    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    class_names : list, optional
        Names of classes
    title : str, default='Confusion Matrix'
        Plot title
    ax : matplotlib.axes, optional
        Axes to plot on
    """
    cm = confusion_matrix(y_true, y_pred)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(title)
    
    return ax


def plot_roc_curve(y_true, y_proba, class_names=None, title='ROC Curve', ax=None):
    """
    Plot ROC curve.
    
    Parameters:
    -----------
    y_true : array-like
        True labels
    y_proba : array-like
        Predicted probabilities
    class_names : list, optional
        Names of classes
    title : str, default='ROC Curve'
        Plot title
    ax : matplotlib.axes, optional
        Axes to plot on
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    unique_classes = np.unique(y_true)
    n_classes = len(unique_classes)
    
    if n_classes == 2:
        # Binary classification
        fpr, tpr, _ = roc_curve(y_true, y_proba[:, 1])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
    else:
        # Multi-class: plot ROC for each class
        for i, cls in enumerate(unique_classes):
            y_true_binary = (y_true == cls).astype(int)
            if len(np.unique(y_true_binary)) == 2:
                fpr, tpr, _ = roc_curve(y_true_binary, y_proba[:, i])
                roc_auc = auc(fpr, tpr)
                label = class_names[i] if class_names else f'Class {cls}'
                ax.plot(fpr, tpr, label=f'{label} (AUC = {roc_auc:.2f})')
    
    ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    return ax


def plot_learning_curve(train_scores, val_scores, param_values, param_name, 
                       title='Learning Curve', ax=None):
    """
    Plot learning curve.
    
    Parameters:
    -----------
    train_scores : array-like
        Training scores for different parameter values
    val_scores : array-like
        Validation scores for different parameter values
    param_values : array-like
        Parameter values
    param_name : str
        Name of the parameter
    title : str, default='Learning Curve'
        Plot title
    ax : matplotlib.axes, optional
        Axes to plot on
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(param_values, train_scores, 'o-', label='Training Score', linewidth=2)
    ax.plot(param_values, val_scores, 'o-', label='Validation Score', linewidth=2)
    ax.set_xlabel(param_name)
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    return ax


def plot_feature_importance(importance_dict, title='Feature Importance', ax=None, top_n=10):
    """
    Plot feature importance.
    
    Parameters:
    -----------
    importance_dict : dict
        Dictionary mapping feature names to importance scores
    title : str, default='Feature Importance'
        Plot title
    ax : matplotlib.axes, optional
        Axes to plot on
    top_n : int, default=10
        Number of top features to display
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Sort by importance
    sorted_features = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
    
    # Get top N features
    top_features = sorted_features[:top_n]
    features, importances = zip(*top_features)
    
    # Create horizontal bar plot
    y_pos = np.arange(len(features))
    ax.barh(y_pos, importances)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(features)
    ax.set_xlabel('Importance Score')
    ax.set_title(title)
    ax.invert_yaxis()  # Top feature at top
    ax.grid(True, alpha=0.3, axis='x')
    
    return ax


def print_metrics(metrics, model_name='Model'):
    """
    Print metrics in a formatted way.
    
    Parameters:
    -----------
    metrics : dict
        Dictionary of metrics
    model_name : str, default='Model'
        Name of the model
    """
    print(f"\n{model_name} Metrics:")
    print("-" * 40)
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1-Score:  {metrics['f1_score']:.4f}")
    if 'roc_auc' in metrics and metrics['roc_auc'] is not None:
        print(f"ROC-AUC:   {metrics['roc_auc']:.4f}")
    print("-" * 40)


### Import Custom Modules

In [ ]:
from knn_classifier import KNNClassifier
from random_forest import RandomForest
from naive_bayes import NaiveBayes
from evaluation_utils import (
    calculate_metrics, plot_confusion_matrix, plot_roc_curve,
    plot_learning_curve, plot_feature_importance, print_metrics
)

print("✅ All custom modules imported successfully!")

## Step 4: Load and Preprocess Data

### Load Dataset

In [ ]:
# Load dataset
try:
    df = pd.read_csv('heart.csv')
except FileNotFoundError:
    import urllib.request
    url = 'https://raw.githubusercontent.com/plotly/datasets/master/heart.csv'
    df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("\nDataset Statistics:")
print(df.describe())

### Check for Missing Values

In [ ]:
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print("Missing Values:")
    print(missing_values[missing_values > 0])
else:
    print("✅ No missing values found!")

### Exploratory Data Analysis

In [ ]:
# Identify target variable
target_col = df.columns[-1] if 'target' not in df.columns else 'target'
print(f"Target variable: {target_col}")

# Check class distribution
print("\nClass Distribution:")
print(df[target_col].value_counts())
print(f"\nClass Distribution (%):")
print(df[target_col].value_counts(normalize=True) * 100)

# Visualize class distribution
plt.figure(figsize=(8, 6))
df[target_col].value_counts().plot(kind='bar')
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Separate numeric and categorical features
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_features:
    numeric_features.remove(target_col)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

### Data Preprocessing

In [ ]:
# Handle missing values
for feature in numeric_features:
    if df[feature].isnull().any():
        df[feature].fillna(df[feature].median(), inplace=True)

for feature in categorical_features:
    if df[feature].isnull().any():
        df[feature].fillna(df[feature].mode()[0], inplace=True)

print("✅ Missing values handled.")

In [ ]:
# Encode categorical variables
label_encoders = {}
df_processed = df.copy()

for feature in categorical_features:
    le = LabelEncoder()
    df_processed[feature] = le.fit_transform(df[feature])
    label_encoders[feature] = le

if len(categorical_features) == 0:
    print("✅ No categorical features to encode.")
else:
    print(f"✅ Encoded {len(categorical_features)} categorical features.")

In [ ]:
# Prepare features and target
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature names: {list(X.columns)}")

In [ ]:
# Split data: 70% training, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nClass distribution in training: {np.bincount(y_train)}")
print(f"Class distribution in validation: {np.bincount(y_val)}")
print(f"Class distribution in test: {np.bincount(y_test)}")

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("✅ Features standardized.")
feature_names = list(X.columns)

## Step 5: Model Training and Evaluation

### 1. k-NN Classifier

#### Hyperparameter Tuning

In [ ]:
# Tune k-NN
print("="*60)
print("Tuning k-NN Classifier")
print("="*60)

k_values = [1, 3, 5, 7, 10]
knn_results = []

for metric in ['euclidean', 'manhattan']:
    for k in k_values:
        for weighted in [True, False]:
            knn = KNNClassifier(k=k, distance_metric=metric, weighted=weighted)
            knn.fit(X_train_scaled, y_train)
            y_pred_val = knn.predict(X_val_scaled)
            metrics = calculate_metrics(y_val, y_pred_val)
            
            knn_results.append({
                'k': k,
                'metric': metric,
                'weighted': weighted,
                'accuracy': metrics['accuracy'],
                'f1_score': metrics['f1_score']
            })
            
            print(f"k={k:2d}, metric={metric:10s}, weighted={str(weighted):5s}: "
                  f"Acc={metrics['accuracy']:.4f}, F1={metrics['f1_score']:.4f}")

knn_results_df = pd.DataFrame(knn_results)
best_knn_idx = knn_results_df['f1_score'].idxmax()
best_knn_params = knn_results_df.iloc[best_knn_idx]

print(f"\nBest k-NN configuration:")
print(f"k={int(best_knn_params['k'])}, metric={best_knn_params['metric']}, "
      f"weighted={best_knn_params['weighted']}")
print(f"Validation F1-Score: {best_knn_params['f1_score']:.4f}")

#### Train and Evaluate Best k-NN Model

In [ ]:
# Train best k-NN model
knn_best = KNNClassifier(
    k=int(best_knn_params['k']),
    distance_metric=best_knn_params['metric'],
    weighted=best_knn_params['weighted']
)
knn_best.fit(X_train_scaled, y_train)

y_pred_test_knn = knn_best.predict(X_test_scaled)
y_proba_test_knn = knn_best.predict_proba(X_test_scaled)
metrics_test_knn = calculate_metrics(y_test, y_pred_test_knn, y_proba_test_knn)

print_metrics(metrics_test_knn, "k-NN (Test Set)")

In [ ]:
# Learning curve for k-NN
knn_train_scores = []
knn_val_scores = []

for k in k_values:
    knn_temp = KNNClassifier(k=k, distance_metric=best_knn_params['metric'],
                            weighted=best_knn_params['weighted'])
    knn_temp.fit(X_train_scaled, y_train)
    train_pred = knn_temp.predict(X_train_scaled)
    val_pred = knn_temp.predict(X_val_scaled)
    knn_train_scores.append(calculate_metrics(y_train, train_pred)['accuracy'])
    knn_val_scores.append(calculate_metrics(y_val, val_pred)['accuracy'])

plt.figure(figsize=(10, 6))
plot_learning_curve(knn_train_scores, knn_val_scores, k_values, 'k',
                   'k-NN Learning Curve')
plt.savefig('knn_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

### 2. Random Forest Classifier

#### Hyperparameter Tuning

In [ ]:
# Tune Random Forest
print("="*60)
print("Tuning Random Forest Classifier")
print("="*60)

max_depths = [3, 5, 7, 10, None]
rf_results = []

for criterion in ['gini', 'entropy']:
    for max_depth in max_depths:
        rf = RandomForest(n_trees=5, max_depth=max_depth, criterion=criterion, random_state=42)
        rf.fit(X_train, y_train)
        y_pred_val = rf.predict(X_val)
        metrics = calculate_metrics(y_val, y_pred_val)
        
        rf_results.append({
            'max_depth': max_depth if max_depth else 'None',
            'criterion': criterion,
            'accuracy': metrics['accuracy'],
            'f1_score': metrics['f1_score']
        })
        
        depth_str = str(max_depth) if max_depth else 'None'
        print(f"max_depth={depth_str:5s}, criterion={criterion:8s}: "
              f"Acc={metrics['accuracy']:.4f}, F1={metrics['f1_score']:.4f}")

rf_results_df = pd.DataFrame(rf_results)
best_rf_idx = rf_results_df['f1_score'].idxmax()
best_rf_params = rf_results_df.iloc[best_rf_idx]

print(f"\nBest Random Forest configuration:")
print(f"max_depth={best_rf_params['max_depth']}, criterion={best_rf_params['criterion']}")
print(f"Validation F1-Score: {best_rf_params['f1_score']:.4f}")

#### Train and Evaluate Best Random Forest Model

In [ ]:
# Train best Random Forest model
max_depth_best = None if best_rf_params['max_depth'] == 'None' else int(best_rf_params['max_depth'])
rf_best = RandomForest(
    n_trees=5,
    max_depth=max_depth_best,
    criterion=best_rf_params['criterion'],
    random_state=42
)
rf_best.fit(X_train, y_train)

y_pred_test_rf = rf_best.predict(X_test)
y_proba_test_rf = rf_best.predict_proba(X_test)
metrics_test_rf = calculate_metrics(y_test, y_pred_test_rf, y_proba_test_rf)

print_metrics(metrics_test_rf, "Random Forest (Test Set)")

In [ ]:
# Feature importance
rf_importance = rf_best.get_feature_importance(feature_names)
plt.figure(figsize=(10, 6))
plot_feature_importance(rf_importance, 'Random Forest Feature Importance')
plt.savefig('rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Learning curve for Random Forest
rf_train_scores = []
rf_val_scores = []

for md in [3, 5, 7, 10, 15]:
    rf_temp = RandomForest(n_trees=5, max_depth=md,
                          criterion=best_rf_params['criterion'],
                          random_state=42)
    rf_temp.fit(X_train, y_train)
    train_pred = rf_temp.predict(X_train)
    val_pred = rf_temp.predict(X_val)
    rf_train_scores.append(calculate_metrics(y_train, train_pred)['accuracy'])
    rf_val_scores.append(calculate_metrics(y_val, val_pred)['accuracy'])

plt.figure(figsize=(10, 6))
plot_learning_curve(rf_train_scores, rf_val_scores, [3, 5, 7, 10, 15], 'Max Depth',
                   'Random Forest Learning Curve')
plt.savefig('rf_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

### 3. Naive Bayes Classifier

In [ ]:
# Train Naive Bayes
print("="*60)
print("Training Naive Bayes Classifier")
print("="*60)

nb = NaiveBayes()
nb.fit(X_train_scaled, y_train)

y_pred_test_nb = nb.predict(X_test_scaled)
y_proba_test_nb = nb.predict_proba(X_test_scaled)
metrics_test_nb = calculate_metrics(y_test, y_pred_test_nb, y_proba_test_nb)

print_metrics(metrics_test_nb, "Naive Bayes (Test Set)")

## Step 6: Visualizations and Comparison

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_confusion_matrix(y_test, y_pred_test_knn, title='k-NN Confusion Matrix', ax=axes[0])
plot_confusion_matrix(y_test, y_pred_test_rf, title='Random Forest Confusion Matrix', ax=axes[1])
plot_confusion_matrix(y_test, y_pred_test_nb, title='Naive Bayes Confusion Matrix', ax=axes[2])

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

### ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_roc_curve(y_test, y_proba_test_knn, title='k-NN ROC Curve', ax=axes[0])
plot_roc_curve(y_test, y_proba_test_rf, title='Random Forest ROC Curve', ax=axes[1])
plot_roc_curve(y_test, y_proba_test_nb, title='Naive Bayes ROC Curve', ax=axes[2])

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

### Performance Comparison

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
model_names = ['k-NN', 'Random Forest', 'Naive Bayes']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    test_scores = [
        metrics_test_knn[metric],
        metrics_test_rf[metric],
        metrics_test_nb[metric]
    ]
    
    bars = ax.bar(model_names, test_scores, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(f'{metric.replace("_", " ").title()} Comparison (Test Set)')
    ax.set_ylim([0, 1.1])
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}',
               ha='center', va='bottom')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### Summary Table

In [ ]:
# Create summary table
summary_data = []
for name, metrics in [('k-NN', metrics_test_knn), 
                      ('Random Forest', metrics_test_rf), 
                      ('Naive Bayes', metrics_test_nb)]:
    summary_data.append({
        'Model': name,
        'Accuracy': f"{metrics['accuracy']:.4f}",
        'Precision': f"{metrics['precision']:.4f}",
        'Recall': f"{metrics['recall']:.4f}",
        'F1-Score': f"{metrics['f1_score']:.4f}",
        'ROC-AUC': f"{metrics.get('roc_auc', 'N/A')}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY (Test Set)")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)

# Save results
summary_df.to_csv('results_summary.csv', index=False)
print("\n✅ Results saved to 'results_summary.csv'")

# Download results (for Colab)
try:
    from google.colab import files
    print("\nDownloading results...")
    files.download('results_summary.csv')
    files.download('confusion_matrices.png')
    files.download('roc_curves.png')
    files.download('model_comparison.png')
    files.download('knn_learning_curve.png')
    files.download('rf_learning_curve.png')
    files.download('rf_feature_importance.png')
    print("\n✅ All files downloaded!")
except:
    print("\nℹ️  Not in Colab - files saved to current directory")

---
## ✅ Analysis Complete!

All models have been trained, evaluated, and compared. Check the visualizations and summary table above.